In [1]:
import MDAnalysis as mda
import nglview as nv
from MDAnalysis.analysis import rms, distances, align, pca, rms
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle

/apps/gent/RHEL9/cascadelake-ib/software/Biopython/1.85-foss-2025a/lib/python3.13/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [4]:
# Load trajectories (.xtc) and topology file (.tpr)
trajfiles1= ("/scratch/gent/vo/000/gvo00003/vsc48847/apo_protein/charmm-gui-7064934746/gromacs/Rep1/step7_production_rep1_final.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/apo_protein/charmm-gui-7064934746/gromacs/step7_production_rep2_final.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/apo_protein/charmm-gui-7064934746/gromacs/step7_production_rep3_final.xtc",
            )
topfile1 = ("/scratch/gent/vo/000/gvo00003/vsc48847/apo_protein/charmm-gui-7064934746/gromacs/Rep1/step7_production_rep1.tpr")
trajfiles2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep2.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep3.xtc",
            )
topfile2 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDN_bound/gromacs/step7_production_rep1.tpr")
trajfiles3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1.xtc", 
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep2.xtc",
             "/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep3.xtc",
            )
topfile3 = ("/scratch/gent/vo/000/gvo00003/vsc48847/BDQ_bound/gromacs/step7_production_rep1.tpr")
apo = mda.Universe(topfile1, trajfiles1)
bdn = mda.Universe(topfile2, trajfiles2)
bdq = mda.Universe(topfile3, trajfiles3)

159 LEU
160 LEU
161 ALA
162 PRO
163 ILE
291 PHE
294 VAL
295 GLY
298 GLU
299 ALA
301 TYR
302 PHE
305 LEU
384 ILE
1065
196 PHE
204 PRO
207 ILE
211 PHE
504 ALA
508 ASP
541 GLU
542 ALA
544 TYR
545 PHE
548 LEU
549 ALA
620 LEU
623 ALA
624 ALA
626 PHE
627 ILE
1066
585 ALA
589 ASP
622 GLU
625 TYR
626 PHE
629 LEU
701 LEU
704 ALA
705 ALA
708 ILE
1067
666 ALA
670 ASP
696 PHE
699 VAL
703 GLU
706 TYR
707 PHE
778 ILE
781 GLY
782 LEU
783 VAL
785 ALA
786 ALA
788 PHE
789 ILE
1068
747 ALA
751 ASP
780 VAL
781 GLY
784 GLU
787 TYR
788 PHE
791 LEU
863 LEU
866 ALA
867 ALA
870 ILE
1069
821 MET
828 ALA
832 ASP
862 GLY
865 GLU
866 ALA
868 TYR
869 PHE
872 LEU
944 LEU
947 ALA
948 ALA
951 ILE
1070
296 LEU
299 ALA
300 ALA
302 PHE
303 ILE
909 ALA
913 ASP
946 GLU
949 TYR
950 PHE
953 LEU
<AtomGroup [<Atom 4094: N of type NH1 of resname ASP, resid 265 and segid seg_1_PROC>, <Atom 4095: HN of type H of resname ASP, resid 265 and segid seg_1_PROC>, <Atom 4096: CA of type CT1 of resname ASP, resid 265 and segid seg_1_PROC

In [ ]:
def compute_rmsf(universe, topfile, trajfiles, label, workdir):
    import MDAnalysis as mda
    from MDAnalysis.analysis import align, rms
    import numpy as np

    ca = "protein and name CA"

    # ===== Average structure =====
    avg = align.AverageStructure(
        universe,
        universe,
        select=ca,
        ref_frame=0
    ).run(step=10)

    ref = avg.results.universe

    ref.atoms.write(f"{workdir}/avg_{label}.pdb")

    # ===== Align trajectory =====
    aligned_xtc = f"{workdir}/aligned_{label}.xtc"

    align.AlignTraj(
        universe,
        ref,
        select=ca,
        filename=aligned_xtc,
        in_memory=False
    ).run(step=10)

    # ===== Reload aligned =====
    u_aligned = mda.Universe(topfile, aligned_xtc)

    # ===== RMSF =====
    R = rms.RMSF(u_aligned.select_atoms(ca)).run()

    rmsf_vals = R.results.rmsf
    resids = u_aligned.select_atoms(ca).resids

    return resids, rmsf_vals

In [ ]:
workdir= "/scratch/gent/vo/000/gvo00003/vsc48847/rmsf_analysis"
res_apo, rmsf_apo = compute_rmsf(apo, topfile1, trajfiles1, "apo", workdir)
res_bdn, rmsf_bdn = compute_rmsf(bdn, topfile2, trajfiles2, "bdn", workdir)
res_bdq, rmsf_bdq = compute_rmsf(bdq, topfile3, trajfiles3, "bdq", workdir)

In [ ]:
plt.figure(figsize=(14,6))

# ============================================================
# RMSF CURVES
# ============================================================

# APO
plt.plot(
    res_apo,
    rmsf_apo,
    color="#6baed6",
    linewidth=2,
    label="apo"
)

# BDN
plt.plot(
    res_bdn,
    rmsf_bdn,
    color="#31a354",
    linewidth=2,
    label="bdn"
)

# BDQ
plt.plot(
    res_bdq,
    rmsf_bdq,
    color="#cb181d",
    linewidth=2,
    label="bdp"
)

# ============================================================
# C-RING REGION
# ============================================================

plt.axvspan(
    238,
    966,
    color="lightblue",
    alpha=0.15
)

# ============================================================
# BINDING SITES
# Asp residues at helix starts omitted for clarity
# ============================================================

binding_sites = {

    "Leading site": {
        "color": "#1f78b4",
        "residues": [
            298, 299, 301, 302, 305,
            377, 380, 381, 384,
            160, 162, 163
        ]
    },

    "Lagging site": {
        "color": "#33a02c",
        "residues": [
            207, 208, 211,
            541, 542, 544, 545, 548,
            620, 623, 624, 627
        ]
    },

    "Site 3": {
        "color": "#ff7f00",
        "residues": [
            622, 623, 625, 626, 629,
            701, 704, 705, 708
        ]
    },

    "Site 4": {
        "color": "#6a3d9a",
        "residues": [
            703, 704, 706, 707, 710,
            782, 785, 786, 789
        ]
    },

    "Site 5": {
        "color": "#e31a1c",
        "residues": [
            784, 785, 787, 788, 791,
            863, 866, 867, 870
        ]
    },

    "Site 6": {
        "color": "#b15928",
        "residues": [
            865, 866, 868, 869, 872,
            944, 947, 948, 951
        ]
    },

    "Site 7": {
        "color": "#fb9a99",
        "residues": [
            946, 947, 949, 950, 953,
            296, 299, 300, 303
        ]
    }
}

# ============================================================
# RESIDUE MARKERS
# ============================================================

ymin, ymax = plt.ylim()

marker_y = ymax * 0.985

for site_name, site_data in binding_sites.items():

    residues = site_data["residues"]
    color = site_data["color"]

    for i, resid in enumerate(residues):

        # only first marker gets legend label
        label = site_name if i == 0 else None

        plt.plot(
            resid,
            marker_y,
            marker="|",
            linestyle="None",
            color=color,
            markersize=14,
            markeredgewidth=2.5,
            label=label
        )

# ============================================================
# AXES / LEGEND
# ============================================================

plt.xlabel("Residue")
plt.ylabel("RMSF (Å)")

plt.legend(
    frameon=False,
    fontsize=9,
    ncol=2
)

plt.tight_layout()

# ============================================================
# SAVE
# ============================================================

plt.savefig(
    f"{workdir}/RMSF_all_bis.png",
    dpi=300
)

plt.show()